In [1]:
# =======================================================================
# SCRIPT 1: SUMARIZAÇÃO HIERÁRQUICA — VARIANTE BASE
# BitNet b1.58 sem fine-tuning para condensação
# =======================================================================

# =======================================================================
# CÉLULA 0 — Configurações de memória
# =======================================================================
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_SKIP_ALLOCATOR_WARMUP"] = "1"

In [2]:
# =======================================================================
# CÉLULA 1 — Instalação de dependências
# =======================================================================
!pip uninstall torchvision -y
!pip install -q -U transformers datasets peft accelerate
!pip install -q nltk rouge-score
!pip install -q "torchao>=0.16.0"

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 50.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.3 MB/s eta 0:00:00


In [3]:
# =======================================================================
# CÉLULA 2 — Imports
# =======================================================================
import torch
import pandas as pd
import numpy as np
import json
import nltk
import os
import shutil
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList
from tqdm import tqdm
from google.colab import files
import zipfile

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

In [4]:
# =======================================================================
# CÉLULA 3 — Configurações gerais
# =======================================================================
MODEL_ID = "microsoft/bitnet-b1.58-2B-4T-bf16"
VARIANTE = "base"
OUTPUT_DIR = f"./resultados_hierarquico_{VARIANTE}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Tamanho máximo de cada chunk em tokens
MAX_TOKENS_CHUNK = 1024

# Tamanho máximo do documento condensado para etapa final
MAX_TOKENS_CONDENSADO = 4096

# Frequência do backup automático (a cada N instâncias)
BACKUP_A_CADA = 20

# Retomada de progresso
RETOMAR_DE_BACKUP = True          # Mude para True se estiver retomando
# CAMINHO_BACKUP_RETOMADA = f"./resultados_hierarquico_{VARIANTE}.csv"
# CAMINHO_BACKUP_RETOMADA = "./resultados_hierarquico_base/resultados_base.csv"
CAMINHO_BACKUP_RETOMADA = "./resultados/resultados_base.csv"

SEED = 42


print(f"RETOMAR_DE_BACKUP: {RETOMAR_DE_BACKUP}")
print(f"Arquivo existe: {os.path.exists(CAMINHO_BACKUP_RETOMADA)}")

RETOMAR_DE_BACKUP: True
Arquivo existe: False


In [5]:
# =======================================================================
# CÉLULA 4 — Download e carregamento do QMSum
# =======================================================================
from huggingface_hub import hf_hub_download
import zipfile as zf

print("Baixando QMSum...")
zip_path = hf_hub_download(
    repo_id="tau/scrolls",
    filename="qmsum.zip",
    repo_type="dataset"
)
os.makedirs("./qmsum_data", exist_ok=True)
with zf.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("./qmsum_data")

dados_test = []
with open("./qmsum_data/qmsum/test.jsonl", "r") as f:
    for linha in f:
        item = json.loads(linha)
        if not item.get("input") or not item.get("output"):
            continue
        dados_test.append({
            "input": item["input"],
            "output": item["output"]
        })

print(f"-> {len(dados_test)} instâncias carregadas")

Baixando QMSum...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


qmsum.zip:   0%|          | 0.00/27.3M [00:00<?, ?B/s]

-> 281 instâncias carregadas


In [6]:
# =======================================================================
# CÉLULA 5 — Funções auxiliares
# =======================================================================

class StopOnToken(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = stop_token_ids

    def __call__(self, input_ids, scores, **kwargs):
        for stop_id in self.stop_token_ids:
            if input_ids[0][-1] == stop_id:
                return True
        return False

def get_stopping_criteria(tokenizer):
    stop_sequences = ["###", "##", "**", "Note:", "---"]
    stop_ids = []
    for seq in stop_sequences:
        ids = tokenizer.encode(seq, add_special_tokens=False)
        if ids:
            stop_ids.append(ids[0])
    return StoppingCriteriaList([StopOnToken(stop_ids)])

def limpar_output(texto):
    for seq in ["###", "##", "**", "Note:", "---"]:
        if seq in texto:
            texto = texto[:texto.index(seq)].strip()
    return texto

def gerar_texto(model, tokenizer, prompt, max_new_tokens,
                repetition_penalty, stopping_criteria):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=repetition_penalty,
            stopping_criteria=stopping_criteria
        )

    gerado = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return limpar_output(gerado)

def dividir_em_chunks(texto, tokenizer, max_tokens=1024):
    sentencas = nltk.sent_tokenize(texto)
    chunks = []
    chunk_atual = []
    tokens_atuais = 0

    for sentenca in sentencas:
        tokens_sentenca = len(tokenizer.encode(
            sentenca, add_special_tokens=False
        ))

        if tokens_sentenca > max_tokens:
            if chunk_atual:
                chunks.append(" ".join(chunk_atual))
                chunk_atual = []
                tokens_atuais = 0
            chunks.append(sentenca)
            continue

        if tokens_atuais + tokens_sentenca > max_tokens:
            if chunk_atual:
                chunks.append(" ".join(chunk_atual))
            chunk_atual = [sentenca]
            tokens_atuais = tokens_sentenca
        else:
            chunk_atual.append(sentenca)
            tokens_atuais += tokens_sentenca

    if chunk_atual:
        chunks.append(" ".join(chunk_atual))

    return chunks

def resumir_chunk(model, tokenizer, chunk, stopping_criteria):
    prompt = (
        "### Instruction:\nWrite a concise summary of the following "
        "meeting excerpt.\n\n"
        f"### Excerpt:\n{chunk}\n\n"
        "### Summary:\n"
    )
    return gerar_texto(
        model, tokenizer, prompt,
        max_new_tokens=150,
        repetition_penalty=1.3,
        stopping_criteria=stopping_criteria
    )

def responder_query(model, tokenizer, query,
                    documento_condensado, stopping_criteria):
    prompt = (
        "### Instruction:\nWrite a concise summary answering the question "
        "based on the condensed meeting notes.\n\n"
        f"### Question:\n{query}\n\n"
        f"### Condensed Meeting Notes:\n{documento_condensado}\n\n"
        "### Answer:\n"
    )
    return gerar_texto(
        model, tokenizer, prompt,
        max_new_tokens=256,
        repetition_penalty=1.3,
        stopping_criteria=stopping_criteria
    )

def extrair_query_e_transcricao(input_texto):
    linhas = input_texto.split("\n", 1)
    query = linhas[0].strip()
    transcricao = linhas[1].strip() if len(linhas) > 1 else input_texto
    return query, transcricao

def fazer_backup(output_dir, variante):
    """Salva e baixa o backup acumulativo atual."""
    pasta_backup = f"./backup_temp_{variante}"
    os.makedirs(pasta_backup, exist_ok=True)
    shutil.copytree(output_dir, f"{pasta_backup}/resultados",
                    dirs_exist_ok=True)
    zip_nome = f"./backup_hierarquico_{variante}.zip"
    shutil.make_archive(
        zip_nome.replace(".zip", ""), "zip", pasta_backup
    )
    print(f"\nBackup salvo: {zip_nome} "
          f"({os.path.getsize(zip_nome) / (1024**2):.1f} MB)")
    files.download(zip_nome)

In [7]:
# =======================================================================
# CÉLULA 6 — Carregamento do modelo
# =======================================================================
print("Carregando tokenizador e modelo BitNet base...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa"
)
model.eval()
print("Modelo carregado!")

stopping_criteria = get_stopping_criteria(tokenizer)

Carregando tokenizador e modelo BitNet base...


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.83G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/332 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/199 [00:00<?, ?B/s]

Modelo carregado!


In [ ]:
# =======================================================================
# CÉLULA 7 — Retomada ou início do zero
# =======================================================================
CAMINHO_CSV = os.path.join(OUTPUT_DIR, f"resultados_{VARIANTE}.csv")

if RETOMAR_DE_BACKUP and os.path.exists(CAMINHO_BACKUP_RETOMADA):
    print("Retomando de backup...")

    # Upload do backup
    print("Faça upload do backup para retomada:")
    uploaded = files.upload()
    zip_backup = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_backup, "r") as zip_ref:
        zip_ref.extractall("./")

    df_progresso = pd.read_csv(CAMINHO_BACKUP_RETOMADA)
    indices_processados = set(df_progresso.index.tolist())
    resultados = df_progresso.to_dict("records")
    print(f"-> {len(resultados)} instâncias já processadas. "
          f"Retomando da instância {len(resultados)}...")
else:
    print("Iniciando do zero...")
    resultados = []
    indices_processados = set()

Retomando de backup...
Faça upload do backup para retomada:


AttributeError: 'list' object has no attribute 'upload'

In [8]:
# =======================================================================
# CÉLULA 7B — Retomada ou início do zero
# =======================================================================

from google.colab import files as colab_files
import zipfile
import pandas as pd

print("Faça upload do backup_hierarquico_base.zip")
uploaded = colab_files.upload()
zip_backup = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_backup, "r") as zip_ref:
    zip_ref.extractall("./")

df_progresso = pd.read_csv("./resultados/resultados_base.csv")
indices_processados = set(df_progresso["instancia_idx"].tolist())
resultados = df_progresso.to_dict("records")
print(f"-> {len(resultados)} instâncias já processadas.")
print(f"-> Retomando da instância {max(indices_processados)+1}...")

Faça upload do backup_hierarquico_base.zip


Saving backup_hierarquico_base.zip to backup_hierarquico_base.zip
-> 80 instâncias já processadas.
-> Retomando da instância 80...


In [10]:
# =======================================================================
# CÉLULA 7C — Retomada ou início do zero
# =======================================================================
CAMINHO_CSV = os.path.join(OUTPUT_DIR, f"resultados_{VARIANTE}.csv")
print(f"CSV será salvo em: {CAMINHO_CSV}")

CSV será salvo em: ./resultados_hierarquico_base/resultados_base.csv


In [11]:
# =======================================================================
# CÉLULA 8 — Pipeline principal
# =======================================================================
print(f"\nIniciando pipeline hierárquico — variante: {VARIANTE.upper()}")
print(f"Total de instâncias: {len(dados_test)}")
print(f"Backup automático a cada {BACKUP_A_CADA} instâncias\n")

for i, item in enumerate(tqdm(dados_test, desc=f"Hierárquico ({VARIANTE})")):

    # Pula instâncias já processadas na retomada
    if i in indices_processados:
        continue

    query, transcricao = extrair_query_e_transcricao(item["input"])

    # 1. Divide em chunks respeitando sentenças
    chunks = dividir_em_chunks(transcricao, tokenizer, MAX_TOKENS_CHUNK)

    # 2. Resume cada chunk individualmente
    resumos_chunks = []
    for chunk in chunks:
        resumo_chunk = resumir_chunk(model, tokenizer, chunk, stopping_criteria)
        resumos_chunks.append(resumo_chunk)

    # 3. Concatena os resumos intermediários
    documento_condensado = "\n".join(resumos_chunks)

    # Trunca por sentenças se necessário
    tokens_condensado = len(tokenizer.encode(
        documento_condensado, add_special_tokens=False
    ))
    if tokens_condensado > MAX_TOKENS_CONDENSADO:
        sentencas = nltk.sent_tokenize(documento_condensado)
        doc_truncado = []
        tokens_acc = 0
        for s in sentencas:
            t = len(tokenizer.encode(s, add_special_tokens=False))
            if tokens_acc + t > MAX_TOKENS_CONDENSADO:
                break
            doc_truncado.append(s)
            tokens_acc += t
        documento_condensado = " ".join(doc_truncado)
        tokens_condensado = tokens_acc

    # 4. Responde a query com o documento condensado
    resumo_final = responder_query(
        model, tokenizer, query,
        documento_condensado, stopping_criteria
    )

    # Salva resultado desta instância
    resultados.append({
        "instancia_idx": i,
        "query": query,
        "resumo_referencia": item["output"],
        "n_chunks": len(chunks),
        "tokens_condensado": tokens_condensado,
        "documento_condensado": documento_condensado,
        "resumos_chunks": " ||| ".join(resumos_chunks),
        "resumo_final": resumo_final
    })

    # Salva CSV progressivamente
    pd.DataFrame(resultados).to_csv(CAMINHO_CSV, index=False)

    # Backup automático a cada N instâncias
    processados_nesta_sessao = len(resultados) - len(indices_processados)
    if processados_nesta_sessao % BACKUP_A_CADA == 0:
        print(f"\n[Checkpoint] {len(resultados)} instâncias processadas")
        fazer_backup(OUTPUT_DIR, VARIANTE)

print(f"\nPipeline concluído! Total: {len(resultados)} instâncias")


Iniciando pipeline hierárquico — variante: BASE
Total de instâncias: 281
Backup automático a cada 20 instâncias



Hierárquico (base):   0%|          | 0/281 [00:00<?, ?it/s][transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer 


[Checkpoint] 100 instâncias processadas

Backup salvo: ./backup_hierarquico_base.zip (0.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...


[Checkpoint] 280 instâncias processadas

Backup salvo: ./backup_hierarquico_base.zip (0.3 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Hierárquico (base):  99%|█████████▉| 279/281 [13:13:50<03:47, 113.96s/it][transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence.


Pipeline concluído! Total: 282 instâncias


In [12]:
# =======================================================================
# CÉLULA 9 — Verificação de exemplos
# =======================================================================
df = pd.read_csv(CAMINHO_CSV)
print(f"Total salvo: {len(df)} instâncias\n")

for i in range(min(3, len(df))):
    print(f"\n{'='*60}")
    print(f"INSTÂNCIA {i+1}")
    print(f"Chunks: {df.iloc[i]['n_chunks']} | "
          f"Tokens condensado: {df.iloc[i]['tokens_condensado']}")
    print(f"\nQUERY:\n{df.iloc[i]['query']}")
    print(f"\nREFERÊNCIA:\n{df.iloc[i]['resumo_referencia']}")
    print(f"\nRESUMO FINAL:\n{df.iloc[i]['resumo_final']}")

Total salvo: 282 instâncias


INSTÂNCIA 1
Chunks: 13 | Tokens condensado: 1351

QUERY:
Summarize the discussion about the efficacy of the law.

REFERÊNCIA:
Barry Hughes first stated that children had fewer rights than adults and therefore the law should be enforced to defend physical assault. As such social behavior was not available now, the law should change to reflect that. The discussion then turned to talk about the legal framework and its prosecution.

RESUMO FINAL:
During the eleventh session of the Women's Development Select Committee discussing the Child Abolition of Reasonable Punishment (Childhood Offenses - Wales) Bill, key topics covered included reviewing current protective legislations, debates on reasonable punishment limits amidst changing social values, clarity needed on definitions relating to 'chastisement,' exploring public order provisions differing regionally, examining PIT test amendments, enhancing prosecutor education programs, refining approaches to prosecute

In [13]:
# =======================================================================
# CÉLULA 10 — Backup final completo
# =======================================================================
fazer_backup(OUTPUT_DIR, VARIANTE)
print("Script concluído!")


Backup salvo: ./backup_hierarquico_base.zip (0.3 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Script concluído!
